# Advanced Trees, Graphs, and Hashing

This notebook continues the data structure series with deeper tree operations, graph algorithms, and hash table internals. It is written in a study-friendly, definition-plus-example format.

## 1. Binary Search Tree (BST) Deletion

Deleting a node from a BST has three cases:
- Node is a leaf -> remove directly.
- Node has one child -> replace node with its child.
- Node has two children -> replace node with inorder successor or predecessor.

### Why deletion is important
- Preserves order in the tree.
- Maintains search efficiency.
- Helps implement dynamic sets and maps.

In [ ]:
class BSTNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

def bst_insert(root, value):
    if root is None:
        return BSTNode(value)
    if value < root.value:
        root.left = bst_insert(root.left, value)
    else:
        root.right = bst_insert(root.right, value)
    return root

def find_min(node):
    while node.left is not None:
        node = node.left
    return node

def bst_delete(root, value):
    if root is None:
        return None
    if value < root.value:
        root.left = bst_delete(root.left, value)
    elif value > root.value:
        root.right = bst_delete(root.right, value)
    else:
        if root.left is None:
            return root.right
        elif root.right is None:
            return root.left
        temp = find_min(root.right)
        root.value = temp.value
        root.right = bst_delete(root.right, temp.value)
    return root

def bst_inorder(root):
    return bst_inorder(root.left) + [root.value] + bst_inorder(root.right) if root else []

root = None
for value in [50, 30, 70, 20, 40, 60, 80]:
    root = bst_insert(root, value)
print('Initial inorder:', bst_inorder(root))
root = bst_delete(root, 30)
print('After deleting 30:', bst_inorder(root))
root = bst_delete(root, 50)
print('After deleting 50:', bst_inorder(root))

## 2. Tree Balance and Height

A tree's height affects search performance. Balanced trees keep height low.

### Height of a binary tree
- Height = max height of left and right subtree + 1.
- Balanced trees have height `O(log n)`.
- Unbalanced trees can degrade to `O(n)` for search.

### Example: compute tree height


In [ ]:
def tree_height(node):
    if node is None:
        return 0
    left_height = tree_height(node.left)
    right_height = tree_height(node.right)
    return max(left_height, right_height) + 1

print('Tree height:', tree_height(root))

## 3. Graph Representation and Conversion

Graphs can be represented using adjacency lists or adjacency matrices. Each representation has tradeoffs.

### Adjacency list
- Efficient for sparse graphs.
- Stores only existing edges.
- Fast iteration over neighbors.

### Adjacency matrix
- Efficient for dense graphs.
- Uses O(V^2) space.
- Easy to check whether an edge exists in O(1).

### Conversion example


In [ ]:
nodes = ['A', 'B', 'C', 'D']
adj_list = {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A'], 'D': ['B']}
index = {node: i for i, node in enumerate(nodes)}
matrix = [[0] * len(nodes) for _ in range(len(nodes))]
for node, neighbors in adj_list.items():
    for neighbor in neighbors:
        matrix[index[node]][index[neighbor]] = 1
print('Adjacency list:', adj_list)
print('Adjacency matrix:')
for row in matrix:
    print(row)

## 4. Topological Sort and Cycle Detection

Topological sort orders nodes of a directed acyclic graph (DAG) so that edges go from earlier to later nodes.

### Use-cases
- Task scheduling.
- Build order for dependencies.
- Course prerequisite planning.

### Cycle detection
- A directed graph with a cycle cannot be topologically sorted.
- Use DFS with recursion stack or Kahn's algorithm to detect cycles.

In [ ]:
from collections import deque

graph = {
    'A': ['B'],
    'B': ['C'],
    'C': ['D'],
    'D': []
}

def topological_sort(graph):
    indegree = {node: 0 for node in graph}
    for node in graph:
        for neighbor in graph[node]:
            indegree[neighbor] += 1
    queue = deque([node for node, deg in indegree.items() if deg == 0])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)
    if len(order) != len(graph):
        return None  # cycle detected
    return order

print('Topological order:', topological_sort(graph))

## 5. Hash Table Fundamentals

A hash table stores key-value pairs and uses a hash function to compute an index in an underlying array.

### Key ideas
- Hash function maps keys to integers (indices).
- Collisions occur when different keys map to the same index.
- Collision resolution strategies include chaining and open addressing.

### Python `dict` behavior
- Python dictionaries use open addressing with probe sequences.
- They maintain insertion order since Python 3.7.
- Average time complexity for `get` and `set` is O(1).

### 5.1 Chaining example

Chaining stores multiple key-value pairs in a bucket list for each index.

In [ ]:
class ChainedHashTable:
    def __init__(self, size=7):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        return hash(key) % self.size

    def set(self, key, value):
        index = self._hash(key)
        bucket = self.table[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))

    def get(self, key):
        index = self._hash(key)
        bucket = self.table[index]
        for k, v in bucket:
            if k == key:
                return v
        return None

hash_table = ChainedHashTable()
hash_table.set('apple', 100)
hash_table.set('banana', 200)
hash_table.set('grape', 300)
print('apple ->', hash_table.get('apple'))
print('banana ->', hash_table.get('banana'))
print('grape ->', hash_table.get('grape'))

### 5.2 Open addressing overview

Open addressing stores all entries in the array itself and resolves collisions with probing strategies such as linear probing or quadratic probing.

### Example of a simple linear probe hash table


In [ ]:
class LinearProbeHashTable:
    def __init__(self, size=7):
        self.size = size
        self.keys = [None] * size
        self.values = [None] * size

    def _hash(self, key):
        return hash(key) % self.size

    def set(self, key, value):
        index = self._hash(key)
        start = index
        while self.keys[index] is not None and self.keys[index] != key:
            index = (index + 1) % self.size
            if index == start:
                raise Exception('Hash table is full')
        self.keys[index] = key
        self.values[index] = value

    def get(self, key):
        index = self._hash(key)
        start = index
        while self.keys[index] is not None:
            if self.keys[index] == key:
                return self.values[index]
            index = (index + 1) % self.size
            if index == start:
                break
        return None

lp_hash = LinearProbeHashTable()
lp_hash.set('x', 1)
lp_hash.set('y', 2)
lp_hash.set('z', 3)
print('x ->', lp_hash.get('x'))
print('y ->', lp_hash.get('y'))
print('z ->', lp_hash.get('z'))

## 6. Practical Applications

- Use BSTs for ordered sets, sorted maps, and range queries.
- Use heaps for priority scheduling, top-k selection, and event simulation.
- Use graphs for network analysis, recommendations, and routing.
- Use hash tables for fast lookup, caching, and frequency counting.

### Data science note
- Many machine learning pipelines use hash tables for feature counting and deduplication.
- Graphs are used for social network analysis, knowledge graphs, and recommendation engines.
- Trees appear in decision tree models and hierarchical clustering.

## 7. Summary

This notebook completes the advanced section by covering BST deletion, tree height, graph representations, topological sort, and hash table implementation patterns. Use it to teach deeper algorithmic concepts and real-world data structure applications.